# Phase 0 — EDA: 데이터 구조 탐색

**목표:** `conversations.json`의 구조를 파악해 파서(Phase 1) 설계에 필요한 정보를 얻는다.

**개인정보 처리:**
- `data/` → `.gitignore` (원본 파일 업로드 안 됨)
- `nbstripout` git hook → 커밋 시 셀 출력 자동 제거

---

```
conversations.json
  └─ [ ] conversation
        ├─ uuid, name, created_at, updated_at
        └─ chat_messages[]
              ├─ uuid, sender, text, created_at
              └─ content[]
                    └─ block (type: text | thinking | tool_use | tool_result | token_budget)
```

## 0. 데이터 로드

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path('../data')

with open(DATA_DIR / 'conversations.json', encoding='utf-8') as f:
    conversations = json.load(f)

all_messages = [msg for c in conversations for msg in c['chat_messages']]
all_blocks   = [b for msg in all_messages for b in msg.get('content', [])]

print(f'대화:    {len(conversations)}')
print(f'메시지:  {len(all_messages):,}')
print(f'블록:    {len(all_blocks):,}')

## 1. 최상위 구조 — conversation 필드

In [ ]:
sample_conv = conversations[0]
for key, val in sample_conv.items():
    if key == 'chat_messages':
        print(f'  {key}: [{len(val)}개 메시지]')
    else:
        print(f'  {key}: {str(val)[:60]}')

## 2. 메시지 구조 — chat_message 필드

In [ ]:
sample_msg = all_messages[0]
for key, val in sample_msg.items():
    if key == 'content':
        print(f'  {key}: [{len(val)}개 블록]')
    else:
        print(f'  {key}: {str(val)[:60]}')

In [ ]:
# text 필드 vs content 블록의 text — 같은가 다른가?
# 파서에서 어느 쪽을 써야 할지 결정하기 위해 확인
mismatches = []
for msg in all_messages:
    content_text = ''.join(
        b.get('text', '') for b in msg.get('content', []) if b.get('type') == 'text'
    ).strip()
    top_text = msg.get('text', '').strip()
    if content_text and top_text and content_text != top_text:
        mismatches.append((top_text[:80], content_text[:80]))

print(f'text 필드 ≠ content text 블록: {len(mismatches)}건')
if mismatches:
    print('\n예시:')
    for top, content in mismatches[:2]:
        print(f'  top:     {top}')
        print(f'  content: {content}')
        print()

## 3. 블록 타입별 필드 구조

파서가 각 타입을 처리할 때 어떤 필드를 읽어야 하는지 파악한다.

In [ ]:
# 타입별 전체 필드 목록 (실제 데이터에서 나타난 모든 필드)
type_fields = defaultdict(set)
type_counts = Counter()
for b in all_blocks:
    t = b.get('type')
    type_fields[t].update(b.keys())
    type_counts[t] += 1

for t, cnt in type_counts.most_common():
    fields = sorted(type_fields[t])
    print(f'[{t}] ({cnt:,}개)')
    print(f'  필드: {fields}')
    print()

In [ ]:
# text 블록 예시
text_block = next(b for b in all_blocks if b.get('type') == 'text')
preview = dict(text_block)
preview['text'] = preview['text'][:100] + '...'
print(json.dumps(preview, ensure_ascii=False, indent=2))

In [ ]:
# thinking 블록 — text가 아닌 thinking 필드에 내용이 있음!
thinking_block = next(b for b in all_blocks if b.get('type') == 'thinking')
preview = dict(thinking_block)
preview['thinking'] = preview['thinking'][:100] + '...'
print(json.dumps(preview, ensure_ascii=False, indent=2))

In [ ]:
# tool_use — artifacts 외 어떤 name이 있나?
tool_use_blocks = [b for b in all_blocks if b.get('type') == 'tool_use']
print('tool_use name 분포:')
for name, cnt in Counter(b.get('name') for b in tool_use_blocks).most_common():
    print(f'  {name}: {cnt}')

In [ ]:
# tool_use artifacts 예시
artifact_block = next(b for b in tool_use_blocks if b.get('name') == 'artifacts')
preview = dict(artifact_block)
preview['input'] = dict(preview['input'])
if 'content' in preview['input']:
    preview['input']['content'] = preview['input']['content'][:100] + '...'
print(json.dumps(preview, ensure_ascii=False, indent=2))

In [ ]:
# artifacts가 아닌 tool_use 예시 (웹검색 등)
other_tool = next((b for b in tool_use_blocks if b.get('name') != 'artifacts'), None)
if other_tool:
    preview = dict(other_tool)
    if 'input' in preview:
        preview['input'] = str(preview['input'])[:100]
    print(json.dumps(preview, ensure_ascii=False, indent=2))
else:
    print('artifacts 외 tool_use 없음')

In [ ]:
# tool_result — content 안에 또 타입이 있음
tool_result_blocks = [b for b in all_blocks if b.get('type') == 'tool_result']
inner_types = Counter(
    item.get('type') for b in tool_result_blocks for item in b.get('content', [])
)
print('tool_result.content 내부 타입:')
for t, cnt in inner_types.most_common():
    print(f'  {t}: {cnt:,}')

error_cnt = sum(1 for b in tool_result_blocks if b.get('is_error'))
print(f'\nis_error=True: {error_cnt}개 / {len(tool_result_blocks)}개')

In [ ]:
# token_budget — 파서에서 무시해도 되는지 확인
token_budget_blocks = [b for b in all_blocks if b.get('type') == 'token_budget']
print(f'token_budget 블록 수: {len(token_budget_blocks)}')
print(json.dumps(token_budget_blocks[0], ensure_ascii=False, indent=2))

## 4. 엣지 케이스 확인

In [ ]:
# content가 없는 메시지
no_content = [msg for msg in all_messages if not msg.get('content')]
print(f'content 없는 메시지: {len(no_content)}개')

# 알 수 없는 블록 타입
known = {'text', 'thinking', 'tool_use', 'tool_result', 'token_budget'}
unknown = [b for b in all_blocks if b.get('type') not in known]
print(f'알 수 없는 블록 타입: {len(unknown)}개')
if unknown:
    print(Counter(b.get('type') for b in unknown))

## 5. artifact 분석 — RAG 포함 여부 결정

In [ ]:
artifacts = [
    {
        'artifact_type': b['input'].get('type', ''),
        'title': b['input'].get('title', ''),
        'char_len': len(b['input'].get('content', '')),
    }
    for b in tool_use_blocks if b.get('name') == 'artifacts'
]

print(f'artifact 수: {len(artifacts)}')
print()
print('타입 분포:')
for t, cnt in Counter(a['artifact_type'] for a in artifacts).most_common():
    print(f'  {t}: {cnt}')

In [ ]:
import statistics
lengths = sorted(a['char_len'] for a in artifacts)
n = len(lengths)

print('artifact content 길이 (문자 수):')
print(f'  최소:   {min(lengths):,}')
print(f'  평균:   {int(statistics.mean(lengths)):,}')
print(f'  p75:    {lengths[int(n*0.75)]:,}')
print(f'  p90:    {lengths[int(n*0.90)]:,}')
print(f'  최대:   {max(lengths):,}')
print()
print('토큰 추정 (÷3.5):')
print(f'  평균: ~{int(statistics.mean(lengths)/3.5):,}토큰')
print(f'  p90:  ~{int(lengths[int(n*0.90)]/3.5):,}토큰')
print(f'  최대: ~{int(max(lengths)/3.5):,}토큰')

## 6. 파서 설계 결론

위 탐색 결과를 바탕으로 직접 채워보세요.

In [ ]:
print('=== 파서 설계 메모 ===')
print()
print('Q1. text 필드 vs content 블록, 파서에서 뭘 써야 할까?')
print('  → ')
print()
print('Q2. thinking 블록은 text가 아닌 thinking 필드에 있다. 파서에서 어떻게 처리?')
print('  → ')
print()
print('Q3. token_budget 블록은 RAG 컨텍스트에서 제외해도 되나?')
print('  → ')
print()
print('Q4. artifact를 RAG에 포함할까? 포함한다면 어떤 방식으로?')
print('  → ')
print()
print('Q5. 알 수 없는 블록 타입이 미래에 생기면 파서가 어떻게 해야 할까?')
print('  → ')